In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Log-Log Plots

We already met *log plots* when we studied Newton-Raphson: put the $y$ axis on a log scale and every step up the axis *multiplies* the value by a fixed amount instead of adding to it.  That was just the thing for watching a mistake shrink across many orders of magnitude.

Here's the sibling trick.  Put *both* axes on a log scale and you get a *log-log plot*.  Log-log plots are the tool for answering questions like "does this error scale like $\Delta x$, $\Delta x^2$, or some other power of $\Delta x$?"

## The Problem With Regular Axes

Suppose somebody hands you three functions, $x$, $x^2$, and $x^3$, and you want to tell them apart near $x=0$.

In [ ]:
x = np.linspace(0, 1, 1000)

fig, ax = plt.subplots()
for p in (1, 2, 3):
    ax.plot(x, x**p, label=f'$x^{p}$')

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.legend()

The line is easy to spot.  But $x^2$ and $x^3$?  They both hug zero and then curve upwards.  If I erased the legend and showed you just one of them, could you tell me which power it is without some careful calculation?

And the region near zero is exactly the region we care about most: when we study numerical errors we want to know what happens as $\Delta x \to 0$.

## Take the Log of Both Sides

If $y(x) = x^p$ you can reveal the power by plotting $\log(y)$ against $\log(x)$.

If we take the log of both sides of the equation,
\begin{align}
    y(x) &= x^p
    \\
    \log y &= \log x^p = p \log x
\end{align}
by the rules for logarithms.  In other words, if we plot $y$ and $x$ *both* on a log scale we get a line with slope $p$.

A *power* turns into a *slope*, and slopes are easy to see.

In [ ]:
x = np.logspace(-4, 0, 1000)

fig, ax = plt.subplots()
for p in (1, 2, 3):
    ax.plot(x, x**p, label=f'$x^{p}$')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.legend()

No more confusion: three straight lines with three different slopes.

Notice what happened to the small-$x$ region, too.  Before it was squished into the corner; now it gets most of the plot.  On a log axis each *decade* (each factor of 10) gets the same amount of room, whether it's the one from 0.1 to 1 or the one from 0.0001 to 0.001.

By the way, `np.logspace(-4, 0, 1000)` gives 1000 points from $10^{-4}$ to $10^{0}$ that are evenly spaced *on a log axis*.  It's what you want here; `np.linspace` would pile nearly all its points into the last decade.

## What About a Number Out Front?

Real life is rarely as clean as $y = x^p$.  More often there's a constant out front, $y = c x^p$.  Take the log again,
\begin{align}
    \log y &= \log c + p \log x.
\end{align}
That's *still* a line with slope $p$!  The constant doesn't change the slope at all, it just slides the whole line up or down by $\log c$.

This is great news for us.  When we say an error scales like $\Delta x^p$ we care a lot about $p$ and not very much about the constant, and a log-log plot pulls the two apart: the power is the *slope* and the constant is the *height*.

In [ ]:
x = np.logspace(-4, 4, 1000)

fig, ax = plt.subplots()
ax.plot(x, x**1, label='$x$')
ax.plot(x, 2 * x**1, label='$2x$')
ax.plot(x, x**2, label='$x^2$')
ax.plot(x, x**1 + 7, label='$x + 7$')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.legend()

Three things to see:

 - $x$ and $2x$ are parallel.  Same power, so same slope; the 2 just lifts the line a little.
 - $x^2$ is steeper.  In fact it's exactly twice as steep.
 - $x + 7$ is *not* a straight line!  On regular axes it's a perfectly good line, but it isn't a pure power.  When $x$ is tiny the 7 wins and the curve is flat (slope 0, like $x^0$).  When $x$ is huge the $x$ wins and the curve joins the slope-1 lines.

That last one is worth remembering: a log-log plot shows you whichever power is *winning* in each region.

## Reading Off the Power

You can do it by eye.  Pick a decade along the $x$ axis and count how many decades the curve climbs.  One decade up for one decade over means $p=1$.  Two decades up means $p=2$.

Let's try it on some pretend error data.  Real errors are never a pure power, so let's make ours mostly $\Delta x^2$ with a little bit of $\Delta x^3$ mixed in.

In [ ]:
Delta_x = 1.0 / 2**np.arange(0, 15)
error = 0.3 * Delta_x**2 + 2 * Delta_x**3

fig, ax = plt.subplots()
ax.plot(Delta_x, error, marker='o', linestyle='none', label='pretend error')

# Lines with a known slope to guide the eye.
ax.plot(Delta_x, Delta_x**1, color='gray', alpha=0.5, linestyle='dashed', label='slope 1')
ax.plot(Delta_x, Delta_x**2, color='gray', alpha=0.5, label='slope 2')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Δx')
ax.set_ylabel('error')
ax.legend()

At big $\Delta x$ the points bend upwards, because that's where the $\Delta x^3$ piece is winning.  But as $\Delta x \to 0$ they straighten out and run parallel to the slope-2 guide line.  Second order!

Drawing a guide line with a slope you know is a cheap way to show a scaling.  If your points run parallel to it, you're done.

If you want an actual number, fit a straight line to the *logs*.  Just make sure to use the small-$\Delta x$ points, where the power you care about is the one that's winning.

In [ ]:
small = Delta_x < 1e-2
slope, intercept = np.polyfit(np.log(Delta_x[small]), np.log(error[small]), 1)

print(f'power    ≈ {slope:.3f}')
print(f'constant ≈ {np.exp(intercept):.3f}')

We put in a power of 2 and a constant of 0.3, and that's pretty close to what came back out.  It's not *exactly* right because of the little bit of $\Delta x^3$ we mixed in.

## Which Plot Shows Which Function?

Every kind of axis has a kind of function that it turns into a straight line.

| axes | a straight line means | the slope tells you |
|:--|:--|:--|
| regular | $y = m x + b$ | $m$ |
| log on $y$ only | $y = A e^{k x}$ | $k$ |
| log on both | $y = c x^p$ | $p$ |

So if you suspect an exponential, make a log plot.  If you suspect a power, make a log-log plot.  If you get a straight line, you guessed right!

## Two Warnings

 - **No zeros and no negatives.**  There's no $\log 0$ and no log of a negative number, so those points just won't show up.  That's why we always plot the *absolute value* of an error, $|\text{numerical} - \text{exact}|$.
 - **Use a few decades.**  Lots of curves look straight if you only look at a tiny piece of them.  A line that stays straight over three or four decades is much more convincing than one that covers half a decade.